# PROJECT: Tabular Classifier (PyTorch vs GBDT)

Reach for this when you need: 
- Reference for comparing Neural Nets (MLP) with Decision Trees (XGBoost/RF).
- To implement a complete tabular training script in PyTorch.
- Understanding when to choose DL over classical ML for tables.

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Data Prep (Adult Income Baseline)

| Feature Type | Preprocessing | Logic |
| :--- | :--- | :--- |
| Numerical | `StandardScaler` | Zero-mean/unit-var for NN stability |
| Categorical | `LabelEncoding` | Required for Embedding layers in PyTorch |
| Target | `to_tensor` | mapping class labels to [0, 1] |

In [ ]:
# Pseudo-loading data (Titanic Proxy)
data = pd.DataFrame({
    'age': [22, 38, 26, 35, 35],
    'fare': [7.25, 71.28, 7.92, 53.10, 8.05],
    'class': [3, 1, 3, 1, 3], # Categorical index
    'target': [0, 1, 1, 1, 0]
})

X_num = StandardScaler().fit_transform(data[['age', 'fare']])
X_cat = data[['class']].values
y = data['target'].values

X_train_num, X_test_num, y_train, y_test = train_test_split(X_num, y, test_size=0.2)

## 2. Models: Classical vs NP-PyTorch

| Model | Pros | Cons |
| :--- | :--- | :--- |
| **Random Forest** | Robust to scaling, fast to train | High memory for large ensembles |
| **XGBoost** | High accuracy, scales to millions | Prone to overfitting on noisy data |
| **PyTorch MLP** | Handles multimodal inputs (e.g. text+table) | High sensitivity to scaling/hyperparams |

In [ ]:
# Classical Baseline
rf = RandomForestClassifier().fit(X_train_num, y_train)
print(f"RF Accuracy: {accuracy_score(y_test, rf.predict(X_test_num))}")

# PyTorch MLP for Tabular
class SimpleTabularNet(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 2)
        )
    def forward(self, x): 
        return self.net(x)

model = SimpleTabularNet(2).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3)

### Common Pitfalls
- **Winner's Bias**: Assuming NNs always beat XGBoost. For structured data < 10M rows, XGBoost/LightGBM is frequently the winner.
- **Embedding Size**: For tabular, don't use high-rank embeddings (e.g. 1024). Keep them small (e.g. 10-50).
- **Standardization**: If you don't use `StandardScaler` for the MLP, your weights will likely oscillate/explode during training.

### Key Takeaways
- Always start with a `RandomForest` baseline for tabular data; only reach for PyTorch if multimodal features are available.
- `XGBoost` is the industry workhorse for tabular classification/regression.
- For large-scale deep tabular models, consider `TabNet` or `FT-Transformer` architectures.